In [ ]:
# Cell 1 — mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
TFT = ('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting'
       '/notebooks/TFT Baseline Demand Forecasting/TFT')
print(os.path.isdir(TFT), sorted(os.listdir(TFT)) if os.path.isdir(TFT) else '-')

True ['configs', 'data', 'dataset', 'lightning_logs', 'models', 'predictTFT.py', 'trainTFT.py', 'trainer']


In [ ]:

import os; os.environ['MPLBACKEND'] = 'Agg'
!apt-get -qq update && apt-get -qq install -y python3.10-venv python3.10-dev
!rm -rf /content/tft310 && python3.10 -m venv /content/tft310
!/content/tft310/bin/pip -q install --upgrade pip "setuptools<81" wheel
!/content/tft310/bin/pip -q install torch==1.13.1 --index-url https://download.pytorch.org/whl/cu117
!/content/tft310/bin/pip -q install "pytorch-forecasting==0.10.3" "pytorch-lightning==1.9.5" "numpy==1.23.5" pandas pyarrow datasets
!/content/tft310/bin/python -c "import numpy,torch; print('numpy',numpy.__version__,'| CUDA',torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
numpy 1.23.5 | CUDA True


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/notebooks/TFT Baseline Demand Forecasting/TFT')
print(os.getcwd())

/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/notebooks/TFT Baseline Demand Forecasting/TFT


In [ ]:
# RAW — train
!/content/tft310/bin/python -u trainTFT.py

/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/notebooks/TFT Baseline Demand Forecasting/TFT
4500000
load data cost 40.59492254257202s
dataset generation cost 91.92010498046875
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
model init cost 0.20808029174804688
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 136 K 
3  | prescalers                         | ModuleDict                      | 288   
4  | static_variable_selection          | VariableSelectionNet

In [ ]:
# RAW — predict
!/content/tft310/bin/python -u predictTFT.py

1) loading context parquet...
   context: (4500000, 19)
2) loading test parquet...
   test: (350000, 19)
3) concat...
   concat: (4850000, 19)
4) building TimeSeriesDataSet (this is the heavy step)...
   dataset built OK
5) checkpoint: epoch=4-step=3415.ckpt
6) running predict...
7) predict done, saving...
8) saved 350000 -> TFT_baseline__raw.parquet


In [ ]:
# RECOVERED — train
!/content/tft310/bin/python -u trainTFT.py --demand

/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/notebooks/TFT Baseline Demand Forecasting/TFT
4500000
load data cost 40.242671728134155s
dataset generation cost 92.21212577819824
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
model init cost 0.20053625106811523
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 136 K 
3  | prescalers                         | ModuleDict                      | 288   
4  | static_variable_selection          | VariableSelectionNe

In [ ]:
!grep -n "demand_path\|TRAIN_DATA_PATH\|PRED_DIR\|^TRAIN\|^TEST" trainTFT.py predictTFT.py

trainTFT.py:15:TRAIN_DATA_PATH = os.environ.get(
trainTFT.py:81:        "--demand_path",
trainTFT.py:98:        df = pd.read_parquet(args.demand_path)
trainTFT.py:104:        df = pd.read_parquet(TRAIN_DATA_PATH)
predictTFT.py:10:TRAIN = os.environ.get("FRN_TRAIN_PARQUET",
predictTFT.py:12:TEST = os.environ.get("FRN_TEST_PARQUET",
predictTFT.py:14:PRED_DIR = os.environ.get("FRN_PRED_DIR",
predictTFT.py:17:os.makedirs(PRED_DIR, exist_ok=True)
predictTFT.py:41:    ap.add_argument("--demand_path", default=os.environ.get(
predictTFT.py:46:    ds=loadDataset(dt, a.demand_path)
predictTFT.py:58:    s.to_parquet(f'{PRED_DIR}/TFT_baseline__{iv}.parquet')


In [ ]:
!grep -n -A3 "FRN_TRAIN_PARQUET\|FRN_TEST_PARQUET\|FRN_PRED_DIR\|FRN_DEMAND_PARQUET" trainTFT.py predictTFT.py

trainTFT.py:16:    "FRN_TRAIN_PARQUET",
trainTFT.py-17-    "/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/inputs/freshretailnet_train.parquet"
trainTFT.py-18-)
trainTFT.py-19-
--
trainTFT.py:84:            "FRN_DEMAND_PARQUET",
trainTFT.py-85-            "/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/Latent Demand Recovery/exp/demand/demand.parquet"
trainTFT.py-86-        ),
trainTFT.py-87-        help="demand data path, default '../../latent_demand_recovery/exp/demand/demand.parquet'"
--
predictTFT.py:10:TRAIN = os.environ.get("FRN_TRAIN_PARQUET",
predictTFT.py-11-    "/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/inputs/freshretailnet_train.parquet")
predictTFT.py:12:TEST = os.environ.get("FRN_TEST_PARQUET",
predictTFT.py-13-    "/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/inputs/freshretailnet_test.parquet")
predictTFT.py:14:PRED_DIR = os.environ.get("FRN_PRED_DIR",
predictT

In [ ]:
# RECOVERED — predict
!/content/tft310/bin/python -u predictTFT.py --demand

1) loading context parquet...
   context: (4500000, 20)
2) loading test parquet...
   test: (350000, 19)
3) concat...
   concat: (4850000, 20)
4) building TimeSeriesDataSet (this is the heavy step)...
   dataset built OK
5) checkpoint: epoch=4-step=3415.ckpt
6) running predict...
7) predict done, saving...
8) saved 350000 -> TFT_baseline__recovered.parquet


In [ ]:
import pandas as pd
PRED = '/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/predictions/test'
OLD  = '/content/drive/MyDrive/Colab Notebooks/Model notebooks1/predictions/test'

for iv in ['raw', 'recovered']:
    new = pd.read_parquet(f'{PRED}/TFT_baseline__{iv}.parquet')
    print(iv, len(new), 'rows | mean', round(new.prediction.mean(), 4))
    old = pd.read_parquet(f'{OLD}/TFT_baseline__{iv}.parquet')
    m = new.merge(old, on=['store_id', 'product_id', 'dt'], suffixes=('_n', '_o'))
    print('   vs July: max diff', round((m.prediction_n - m.prediction_o).abs().max(), 6))

raw 350000 rows | mean 1.0947
   vs July: max diff 14.708316
recovered 350000 rows | mean 1.2575
   vs July: max diff 6.298159


In [ ]:
import sys,os
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

In [ ]:
import config as cf, data, predictions

In [ ]:
import config as cf, data, predictions
import pandas as pd
NEW = f'{cf.PRED_TEST}'
OLD = '/content/drive/MyDrive/Colab Notebooks/Model notebooks1/predictions/test'

new = pd.read_parquet(f'{NEW}/TFT_baseline__recovered.parquet')
old = pd.read_parquet(f'{OLD}/TFT_baseline__recovered.parquet')
m = new.merge(old, on=['store_id','product_id','dt'], suffixes=('_n','_o'))
d = (m.prediction_n - m.prediction_o).abs()

print('mean diff  ', round(d.mean(), 4))
print('median diff', round(d.median(), 4))
print('p99 diff   ', round(d.quantile(0.99), 4))
print('rows > 1.0 ', int((d > 1).sum()), 'of', len(d))

mean diff   0.1256
median diff 0.078
p99 diff    0.8782
rows > 1.0  2454 of 350000


In [ ]:
import importlib, data
importlib.reload(data)
print([x for x in dir(data) if not x.startswith('_')])

['array_cols', 'build', 'cf', 'check', 'load', 'np', 'pd', 'splittbl']


In [ ]:
import evaluate, pandas as pd

t = evaluate.truth('test')

def per_series_labels(folder, model):
    out = {}
    for iv in ['raw', 'recovered']:
        p = pd.read_parquet(f'{folder}/{model}__{iv}.parquet')[
            ['store_id', 'product_id', 'dt', 'prediction']]
        m = p.merge(t, on=['store_id', 'product_id', 'dt'], how='inner')
        w = (m.groupby(['store_id', 'product_id'])
               .apply(lambda g: (g.prediction - g.sale_amount).abs().sum()
                                / g.sale_amount.sum()))
        out[iv] = w
    return (out['recovered'] < out['raw']).rename('rec_better')

new_lab = per_series_labels(cf.PRED_TEST, 'TFT_baseline')
old_lab = per_series_labels('/content/drive/MyDrive/Colab Notebooks/Model notebooks1/predictions/test',
                            'TFT_baseline')

both = pd.concat([new_lab.rename('new'), old_lab.rename('old')], axis=1).dropna()
print('series compared :', len(both))
print('labels agree    :', round((both['new'] == both['old']).mean(), 4))
print('recovered better: new', round(both['new'].mean(), 3), '| old', round(both['old'].mean(), 3))

/tmp/ipykernel_5311/3156480293.py:12: RuntimeWarning: divide by zero encountered in scalar divide
  .apply(lambda g: (g.prediction - g.sale_amount).abs().sum()
/tmp/ipykernel_5311/3156480293.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g.prediction - g.sale_amount).abs().sum()
/tmp/ipykernel_5311/3156480293.py:12: RuntimeWarning: divide by zero encountered in scalar divide
  .apply(lambda g: (g.prediction - g.sale_amount).abs().sum()
/tmp/ipykernel_5311/3156480293.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Eithe

series compared : 50000
labels agree    : 0.7168
recovered better: new 0.497 | old 0.463


/tmp/ipykernel_5311/3156480293.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g.prediction - g.sale_amount).abs().sum()


In [ ]:
def per_series(folder, model):
    out = {}
    for iv in ['raw', 'recovered']:
        p = pd.read_parquet(f'{folder}/{model}__{iv}.parquet')[
            ['store_id', 'product_id', 'dt', 'prediction']]
        m = p.merge(t, on=['store_id', 'product_id', 'dt'], how='inner')
        g = m.groupby(['store_id', 'product_id'])
        err = g.apply(lambda x: (x.prediction - x.sale_amount).abs().sum(), include_groups=False)
        act = g['sale_amount'].sum()
        out[iv] = (err / act)[act > 0]          # drop zero-sales series
    d = pd.concat([out['raw'].rename('raw'), out['recovered'].rename('rec')], axis=1).dropna()
    d['label'] = d['rec'] < d['raw']
    d['gap'] = (d['raw'] - d['rec']).abs()
    return d

new = per_series(cf.PRED_TEST, 'TFT_baseline')
old = per_series('/content/drive/MyDrive/Colab Notebooks/Model notebooks1/predictions/test',
                 'TFT_baseline')

b = new[['label', 'gap']].join(old['label'].rename('label_old'), how='inner').dropna()
print('series:', len(b), '| overall agreement:', round((b.label == b.label_old).mean(), 4))

b['band'] = pd.qcut(b['gap'], 4, labels=['smallest gap', 'small', 'large', 'largest gap'])
print(b.groupby('band', observed=True).apply(
    lambda g: pd.Series({'n': len(g), 'agreement': round((g.label == g.label_old).mean(), 3)})))

series: 49872 | overall agreement: 0.7161
                    n  agreement
band                            
smallest gap  12468.0      0.548
small         12468.0      0.654
large         12468.0      0.768
largest gap   12468.0      0.895


/tmp/ipykernel_5311/3435805612.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(b.groupby('band', observed=True).apply(


In [ ]:
b.groupby('band', observed=True).apply(
    lambda g: pd.Series({'n': len(g), 'agreement': (g.label == g.label_old).mean()})
).to_csv(f'{cf.EVALUATION}/label_stability.csv')

/tmp/ipykernel_5311/3937015101.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  b.groupby('band', observed=True).apply(


In [ ]:
!/content/tft310/bin/python -c "import sys, platform; print('Python:', sys.version.split()[0])"
!/content/tft310/bin/pip list 2>/dev/null | grep -Ei "^(torch|numpy|pandas|pytorch-forecasting|pytorch-lightning|pyarrow) "

/bin/bash: line 1: /content/tft310/bin/python: No such file or directory
